# Aplicación. Evaluación de pruebas diagnósticas de laboratorio (Parte II)
Métodos Computacionales en Ingeniería Biomédica

## Actividad 1. Descenso por gradiente
Se busca el mínimo de $J(u) = 2(u-1)^2 + 1$ con el método de descenso por gradiente:
$$u_{k+1} = u_k - \rho \, J'(u_k), \qquad J'(u) = 4(u-1)$$
con $u_0 = 4$ y tasas $\rho = 0.5,\ 0.25,\ 0.1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def J(u):
    return 2 * (u - 1) ** 2 + 1

def dJ(u):
    return 4 * (u - 1)

def descenso_gradiente(u0, rho, n_iter=15, tol=1e-8):
    """Devuelve el historial de u_k generado por descenso de gradiente."""
    historial = [u0]
    u = u0
    for k in range(n_iter):
        u_nuevo = u - rho * dJ(u)
        historial.append(u_nuevo)
        if abs(u_nuevo - u) < tol:
            break
        u = u_nuevo
    return np.array(historial)

In [ ]:
tasas = [0.5, 0.25, 0.1]
resultados = {}

for rho in tasas:
    historial = descenso_gradiente(u0=4.0, rho=rho, n_iter=15)
    resultados[rho] = historial
    print(f"\nrho = {rho}")
    for k, u in enumerate(historial):
        print(f"  u_{k} = {u:.6f}   J(u_{k}) = {J(u):.6f}")

In [ ]:
plt.figure(figsize=(10, 6))
for rho, historial in resultados.items():
    plt.plot(historial, marker='o', label=f"ρ = {rho}")
plt.axhline(1, color='gray', linestyle='--', label='Mínimo (u=1)')
plt.xlabel('Iteración k')
plt.ylabel('u_k')
plt.title('Descenso por gradiente para distintas tasas ρ')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## Actividad 2. Ajuste del vector de pesos de un perceptrón
Se usa la clase `Perceptron` de scikit-learn (mismo método que en `EnsayoPerceptron.ipynb`), arrancando desde el peso inicial $\underline{w}_0 = (-0.9, 1, 1)^T$ que pide el enunciado: primero se entrena una época con pesos en cero solo para inicializar los atributos internos del modelo, y enseguida se sobreescriben con $w_0$ antes de continuar el ajuste con `partial_fit`.

In [ ]:
from sklearn.linear_model import Perceptron

puntos = np.array([
    [0.4, 0.4,  1],
    [0.5, 0.6,  1],
    [0.4, 0.6,  1],
    [0.5, 0.5,  1],
    [0.1, 0.2, -1],
    [0.1, 0.3, -1],
    [0.2, 0.1, -1],
    [0.2, 0.2, -1],
])

X = puntos[:, :2]
y = puntos[:, 2]

In [ ]:
modelo2 = Perceptron(max_iter=1, tol=None, eta0=1.0, shuffle=False, random_state=10)
modelo2.fit(X, y)  # inicializa los atributos internos del modelo (se sobreescriben a continuación)
modelo2.coef_ = np.array([[1.0, 1.0]])
modelo2.intercept_ = np.array([-0.9])

print(f"w_0: b={modelo2.intercept_[0]}, w1={modelo2.coef_[0][0]}, w2={modelo2.coef_[0][1]}\n")

for epoca in range(1, 21):
    modelo2.partial_fit(X, y)
    w1, w2 = modelo2.coef_[0]
    b = modelo2.intercept_[0]
    precision = modelo2.score(X, y)
    print(f"  Época {epoca}: b={b:.4f}  w1={w1:.4f}  w2={w2:.4f}  precisión={precision * 100:.1f}%")
    if precision == 1.0:
        break

print(f"\nVector de pesos final: b={b:.4f}, w1={w1:.4f}, w2={w2:.4f}")

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='red', label='Clase 1 (+1)', s=80)
plt.scatter(X[y == -1, 0], X[y == -1, 1], color='blue', label='Clase 2 (-1)', s=80)

x1_valores = np.linspace(0, 0.7, 100)
x2_valores = -(w1 * x1_valores + b) / w2
plt.plot(x1_valores, x2_valores, color='black', linewidth=2.5, label='Recta del perceptrón')

plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Clasificación por perceptrón (scikit-learn)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## Actividad 3. Datos bivariados: glucosa y hemoglobina glucosilada
La variable `x` de `DatosBivariados.mat` tiene 3 columnas: glucosa (mg/dL), %HbA1c y clase (+1 prediabetes/diabetes, -1 sano).

**Nota:** este archivo no se encontró en la carpeta del notebook. Si no lo tienes, descárgalo del aula virtual del curso y colócalo junto a `clase.ipynb`; mientras tanto, la celda siguiente genera datos **simulados de marcador de posición** (con rangos clínicos plausibles) únicamente para poder ejecutar y probar todo el código. Los resultados numéricos de la actividad real (prevalencia, regresión, sensibilidad, etc.) deben recalcularse con el archivo verdadero.

In [ ]:
import os
import scipy.io
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Perceptron
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_auc_score

In [ ]:
ruta_datos = "DatosBivariados.mat"

if os.path.exists(ruta_datos):
    datos_mat = scipy.io.loadmat(ruta_datos)
    x = datos_mat['x']
    print(f"Datos cargados desde '{ruta_datos}': {x.shape[0]} muestras")
else:
    print("AVISO: no se encontró 'DatosBivariados.mat' en esta carpeta.")
    print("Se generaron datos SIMULADOS de marcador de posición solo para poder probar el código.")
    print("Coloca el archivo real junto a este notebook y vuelve a ejecutar todas las celdas.\n")

    np.random.seed(7)
    n_sanos, n_enfermos = 650, 350

    glucosa_sanos = np.random.normal(90, 8, n_sanos)
    hba1c_sanos = np.random.normal(5.2, 0.3, n_sanos)
    glucosa_enfermos = np.random.normal(135, 25, n_enfermos)
    hba1c_enfermos = np.random.normal(6.8, 0.7, n_enfermos)

    glucosa = np.concatenate([glucosa_sanos, glucosa_enfermos])
    hba1c = np.concatenate([hba1c_sanos, hba1c_enfermos])
    clase = np.concatenate([-np.ones(n_sanos), np.ones(n_enfermos)])

    x = np.column_stack([glucosa, hba1c, clase])
    x = x[np.random.permutation(len(x))]
    print(f"Datos simulados generados: {x.shape[0]} muestras")

glucosa_total = x[:, 0]
hba1c_total = x[:, 1]
clase_total = x[:, 2]

### a. Prevalencia de prediabetes/diabetes

In [ ]:
prevalencia = np.mean(clase_total == 1) * 100
print(f"Prevalencia de prediabetes/diabetes en la muestra: {prevalencia:.2f}%")

### b. Partición en entrenamiento (70%) y prueba (30%), guardadas en archivos separados

In [ ]:
x_train, x_test = train_test_split(
    x, test_size=0.30, random_state=7, stratify=clase_total
)

print(f"Entrenamiento: {x_train.shape[0]} muestras")
print(f"Prueba: {x_test.shape[0]} muestras")

scipy.io.savemat("DatosBivariados_entrenamiento.mat", {"x": x_train})
scipy.io.savemat("DatosBivariados_prueba.mat", {"x": x_test})
print("\nGuardado: DatosBivariados_entrenamiento.mat y DatosBivariados_prueba.mat")

### c. Gráfica de dispersión de los datos de entrenamiento

In [ ]:
glu_tr, hba_tr, cl_tr = x_train[:, 0], x_train[:, 1], x_train[:, 2]

plt.figure(figsize=(8, 6))
plt.scatter(hba_tr[cl_tr == -1], glu_tr[cl_tr == -1], color='blue', label='Sano (-1)', alpha=0.6)
plt.scatter(hba_tr[cl_tr == 1], glu_tr[cl_tr == 1], color='red', label='Prediabetes/Diabetes (+1)', alpha=0.6)
plt.xlabel('Hemoglobina glucosilada (%)')
plt.ylabel('Glucosa en sangre (mg/dL)')
plt.title('Datos de entrenamiento')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

### d. Regresión lineal: glucosa en función de %HbA1c

In [ ]:
reg = LinearRegression()
reg.fit(hba_tr.reshape(-1, 1), glu_tr)

glu_pred = reg.predict(hba_tr.reshape(-1, 1))
mse = np.mean((glu_tr - glu_pred) ** 2)

print(f"glucosa = {reg.coef_[0]:.4f} * HbA1c + ({reg.intercept_:.4f})")
print(f"Error cuadrático medio (entrenamiento): {mse:.4f}")

### e. Perceptrón: sensibilidad, especificidad, exactitud y AUC
Mismo método que en `EnsayoPerceptron.ipynb` (clase `Perceptron` de scikit-learn, extracción de `coef_`/`intercept_`). Las características se estandarizan antes de entrenar, ya que glucosa (mg/dL) y %HbA1c tienen escalas muy distintas, lo que en la práctica impide que el perceptrón converja a una frontera útil.

In [ ]:
X_train_p = np.column_stack([hba_tr, glu_tr])  # columnas: HbA1c, glucosa

scaler = StandardScaler()
X_train_p_esc = scaler.fit_transform(X_train_p)

modelo = Perceptron(max_iter=1000, tol=1e-3, random_state=10)
modelo.fit(X_train_p_esc, cl_tr)

w1, w2 = modelo.coef_[0]
b = modelo.intercept_[0]
print(f"Frontera (variables estandarizadas): {w1:.4f}*HbA1c_z + {w2:.4f}*glucosa_z + {b:.4f} = 0")

In [ ]:
y_pred = modelo.predict(X_train_p_esc)
tn, fp, fn, tp = confusion_matrix(cl_tr, y_pred, labels=[-1, 1]).ravel()

sensibilidad = tp / (tp + fn)
especificidad = tn / (tn + fp)
exactitud = (tp + tn) / (tp + tn + fp + fn)

scores = modelo.decision_function(X_train_p_esc)
auc = roc_auc_score(cl_tr, scores)

print(f"Sensibilidad:  {sensibilidad:.4f}")
print(f"Especificidad: {especificidad:.4f}")
print(f"Exactitud:     {exactitud:.4f}")
print(f"AUC:           {auc:.4f}")

### f. Datos de entrenamiento con la recta del perceptrón y la recta de mínimos cuadrados

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(hba_tr[cl_tr == -1], glu_tr[cl_tr == -1], color='blue', label='Sano (-1)', alpha=0.6)
plt.scatter(hba_tr[cl_tr == 1], glu_tr[cl_tr == 1], color='red', label='Prediabetes/Diabetes (+1)', alpha=0.6)

hba_vals = np.linspace(hba_tr.min(), hba_tr.max(), 100)

# Recta del perceptrón: se desestandariza para graficar en unidades originales
hba_vals_esc = (hba_vals - scaler.mean_[0]) / scaler.scale_[0]
glu_vals_esc = -(b + w1 * hba_vals_esc) / w2
glu_recta_percep = glu_vals_esc * scaler.scale_[1] + scaler.mean_[1]
plt.plot(hba_vals, glu_recta_percep, 'k-', linewidth=2, label='Recta del perceptrón')

# Recta de mínimos cuadrados
glu_recta_mc = reg.predict(hba_vals.reshape(-1, 1))
plt.plot(hba_vals, glu_recta_mc, 'g--', linewidth=2, label='Recta de mínimos cuadrados')

plt.xlabel('Hemoglobina glucosilada (%)')
plt.ylabel('Glucosa en sangre (mg/dL)')
plt.title('Datos de entrenamiento: perceptrón vs. mínimos cuadrados')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()